# Module 01 Lab — Self-Correcting Code Agent

**Course:** AI Agents Teaching Platform  
**Module:** 01 — Prompting & Reasoning  
**Estimated time:** 2–3 hours

---

### What you'll build

An agent that takes a coding task + test suite, generates Python code with an LLM, runs the tests,
reads the failure output, and retries until the tests pass or the retry limit is reached.

### Constraints
- Maximum 3 retry attempts — not negotiable
- The loop must stop if the same error appears on consecutive attempts
- Every retry must include the full error output from the failed tests
- No unhandled exceptions

> **VS Code / local?** See the lab page on the platform for instructions.

## Step 1 — Choose your model

Set `MODEL` to any model string supported by [LiteLLM](https://docs.litellm.ai/docs/providers).

| Provider | Example model string | Key env var |
|---|---|---|
| Anthropic | `claude-haiku-4-5-20251001` | `ANTHROPIC_API_KEY` |
| OpenAI | `gpt-4o-mini` | `OPENAI_API_KEY` |
| Google Gemini | `gemini/gemini-1.5-flash` | `GEMINI_API_KEY` |
| Groq | `groq/llama-3.1-8b-instant` | `GROQ_API_KEY` |

In [ ]:
MODEL = "claude-haiku-4-5-20251001"
print(f"Model: {MODEL}")

## Step 2 — Install dependencies

In [ ]:
%pip install -q litellm pydantic pytest

## Step 3 — API key

Set the environment variable that matches your provider.

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your API key: ")
print("Key set ✓")

## Step 4 — Pydantic models

These models define the structured data contract between every component.
No TODOs here — read them carefully before implementing the TODOs below.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class CodeOutput(BaseModel):
    code: str = Field(description="The complete Python code, no markdown fences")
    explanation: str = Field(max_length=300, description="One-paragraph explanation")

class ValidationResult(BaseModel):
    passed: bool
    error_output: str | None = None
    test_count: int
    failed_count: int

class AgentTrace(BaseModel):
    attempt: int
    code: str
    validation: ValidationResult
    status: Literal["success", "retry", "failed"]

print("Models defined ✓")

## Step 5 — Build the test runner (TODO)

Fill in `run_tests()` so it:
1. Writes `code` → `solution.py` and `test_suite` → `tests.py` in a temp directory
2. Runs `pytest tests.py --tb=short -q` as a subprocess
3. Parses the return code and output into a `ValidationResult`

**Hint:** `result.returncode == 0` means all tests passed. Parse `(\d+) failed` and `(\d+) passed` from stdout with `re.findall`.

In [ ]:
import subprocess
import tempfile
import os
import re

def run_tests(code: str, test_suite: str, timeout: int = 30) -> ValidationResult:
    """Run pytest on generated code. Returns a ValidationResult."""
    with tempfile.TemporaryDirectory() as tmp:
        # TODO 1: write code to solution.py
        # TODO 2: write test_suite to tests.py
        # TODO 3: run pytest with subprocess.run(["pytest", "tests.py", "--tb=short", "-q"],
        #          cwd=tmp, capture_output=True, text=True, timeout=timeout)
        # TODO 4: parse returncode + output into ValidationResult
        # Hint: failed = sum(int(n) for n in re.findall(r"(\d+) failed", output))
        pass

# Quick self-test (will pass once implemented)
# result = run_tests("def add(a, b): return a + b", "from solution import add\ndef test_add(): assert add(2, 3) == 5")
# assert result.passed
# print("run_tests() self-test passed ✓")

<details>
<summary>💡 Stuck? Reveal the run_tests() solution</summary>

```python
def run_tests(code: str, test_suite: str, timeout: int = 30) -> ValidationResult:
    with tempfile.TemporaryDirectory() as tmp:
        with open(os.path.join(tmp, "solution.py"), "w") as f:
            f.write(code)
        with open(os.path.join(tmp, "tests.py"), "w") as f:
            f.write(test_suite)
        try:
            result = subprocess.run(
                ["pytest", "tests.py", "--tb=short", "-q"],
                cwd=tmp, capture_output=True, text=True, timeout=timeout,
            )
        except subprocess.TimeoutExpired:
            return ValidationResult(passed=False, error_output="Test run timed out",
                                    test_count=0, failed_count=0)
        output = result.stdout + result.stderr
        passed = result.returncode == 0
        failed = sum(int(n) for n in re.findall(r"(\d+) failed", output))
        ok = sum(int(n) for n in re.findall(r"(\d+) passed", output))
        return ValidationResult(
            passed=passed,
            error_output=None if passed else output[-3000:],
            test_count=failed + ok,
            failed_count=failed,
        )
```

</details>

## Step 6 — Prompt templates

These are complete — read them and understand the structure before moving on.

In [ ]:
SYSTEM_PROMPT = """
You are an expert Python developer.
Write clean, correct Python code that passes the provided test suite.
Return a JSON object with:
  - "code": the complete Python implementation (no markdown fences)
  - "explanation": a brief explanation of your approach (1 paragraph)
"""

def build_user_prompt(task: str, test_suite: str) -> str:
    return f"""Task:\n{task}\n\nTest suite that your code must pass:\n```python\n{test_suite}\n```\n\nWrite the implementation now."""

def build_retry_prompt(attempt: int, result: ValidationResult) -> str:
    return f"""Your code failed the test suite (attempt {attempt}).\n\nTest failures:\n{result.error_output}\n\nFix the bugs and return corrected code."""

print("Prompts defined ✓")

## Step 7 — Build the agent (TODO)

Fill in `generate_code()` to call the LLM and return a `CodeOutput`.

**LiteLLM message format:**
```python
import json, litellm
response = litellm.completion(model=MODEL, max_tokens=1024, messages=messages)
raw = response.choices[0].message.content
return CodeOutput.model_validate_json(raw)
```

The `run_agent()` loop is mostly complete — just fill in `generate_code()`.

In [ ]:
import json
import logging
import litellm
from pydantic import ValidationError

litellm.drop_params = True
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)

MAX_RETRIES = 3


def generate_code(messages: list) -> CodeOutput:
    """TODO: call LiteLLM and return a CodeOutput.
    Use temperature=0 so retries converge.
    Catch ValidationError and return CodeOutput(code='', explanation=str(e)).
    """
    # TODO
    pass


def run_agent(task: str, test_suite: str) -> list[AgentTrace]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(task, test_suite)}
    ]
    traces = []
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        log.info(f"Attempt {attempt}/{MAX_RETRIES}")
        code_output = generate_code(messages)
        messages.append({"role": "assistant", "content": code_output.model_dump_json()})

        result = run_tests(code_output.code, test_suite)

        status = "success" if result.passed else (
            "failed" if attempt == MAX_RETRIES or result.error_output == last_error
            else "retry"
        )
        traces.append(AgentTrace(attempt=attempt, code=code_output.code,
                                  validation=result, status=status))

        if status in ("success", "failed"):
            log.info(f"Agent finished: {status}")
            return traces

        last_error = result.error_output
        messages.append({"role": "user", "content": build_retry_prompt(attempt, result)})

    return traces

print("Agent loop defined ✓")

<details>
<summary>💡 Stuck? Reveal generate_code() solution</summary>

```python
def generate_code(messages: list) -> CodeOutput:
    response = litellm.completion(model=MODEL, max_tokens=1024,
                                  temperature=0, messages=messages)
    raw = response.choices[0].message.content
    try:
        return CodeOutput.model_validate_json(raw)
    except (ValidationError, Exception) as e:
        return CodeOutput(code="", explanation=f"Model returned invalid output: {e}")
```

Notice: at `temperature=0` the model converges on the same fix. The convergence check
(`last_error == result.error_output`) catches the case where it can't fix the bug.

</details>

## Step 8 — Test your agent

In [ ]:
TASK_1 = "Write a function fibonacci(n) that returns the nth Fibonacci number (0-indexed)."
TEST_1 = """
from solution import fibonacci
def test_base_cases():
    assert fibonacci(0) == 0
    assert fibonacci(1) == 1
def test_sequence():
    assert fibonacci(5) == 5
    assert fibonacci(10) == 55
"""

traces = run_agent(TASK_1, TEST_1)
print(f"\nResult: {traces[-1].status} after {len(traces)} attempt(s)")
assert traces[-1].status in ("success", "failed"), "Must always terminate"
assert len(traces) <= MAX_RETRIES, "Must not exceed MAX_RETRIES"
print("Task 1: test passed ✓")

In [ ]:
TASK_2 = "Write is_palindrome(s) that returns True if s is a palindrome, ignoring case and non-alphanumeric characters."
TEST_2 = """
from solution import is_palindrome
def test_basic(): assert is_palindrome("racecar") == True
def test_case(): assert is_palindrome("RaceCar") == True
def test_punctuation(): assert is_palindrome("A man, a plan, a canal: Panama") == True
def test_empty(): assert is_palindrome("") == True
"""

traces = run_agent(TASK_2, TEST_2)
print(f"Result: {traces[-1].status} after {len(traces)} attempt(s)")
print("Task 2: test passed ✓")

## Step 9 — Experiments

### Experiment A: Break the convergence check
Change `temperature=0` to `temperature=1.0` in `generate_code`. Run TASK_2 multiple times.
Does the agent still converge? What happens to the duplicate-error stopping condition?

### Experiment B: Unsolvable task
Create a task where the test suite contradicts the task description:
```python
TASK_BAD = "Write add(a, b) that returns a + b."
TEST_BAD = "from solution import add\ndef test_wrong(): assert add(2, 3) == 999"
```
Run it. Verify the agent stops at MAX_RETRIES and the convergence check fires.

### Experiment C: Swap the model
Change `MODEL` to `gpt-4o-mini` (set `OPENAI_API_KEY`). Re-run from Step 7 down.
Which function did you have to change? (Answer: none — only the MODEL string.)

In [ ]:
# Scratch cell — use for experiments


## Step 10 — Reflection

1. Name two conditions where the self-correction loop gets stuck and explain how each is bounded.
2. Why is `temperature=0` important for retries?
3. Why does `CodeOutput` use Pydantic instead of plain `json.loads`?
4. In Experiment C, what was the only thing you changed to switch providers?

*(Double-click to edit)*

1. 
2. 
3. 
4. 